### Credit Readiness Score

#### Goal: Compute a 0-850 credit readiness score from 5 transaction-based signals.

#### Signals: Revenue Consistency.Cash Flow Health.Transaction Frequency.Average Transaction Size.Expense Control

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import plotly.graph_objects as go

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

with engine.connect() as conn:
    count = conn.execute(text("SELECT COUNT(*) FROM transactions")).scalar()
print(f"Connected. {count} transactions available for scoring.")

### Score Bands - Nigerian Microfinance Context

#### Aligned to CBN MSME lending guidelines. Max score is 850(same scale as US FICO for familiarity).

In [ ]:
SCORE_BANDS = [
    {"min": 750, "max": 850, "label": "Excellent",  "color": "#16a34a",
     "description": "Strong loan candidate. Ready for ₦5M+ facility."},
    {"min": 650, "max": 749, "label": "Good",        "color": "#65a30d",
     "description": "Good candidate for ₦1M–₦5M working capital loan."},
    {"min": 550, "max": 649, "label": "Fair",        "color": "#ca8a04",
     "description": "Eligible for micro-loan up to ₦500K. Improve consistency."},
    {"min": 450, "max": 549, "label": "Developing",  "color": "#ea580c",
     "description": "3–6 months more transaction history needed."},
    {"min": 0,   "max": 449, "label": "Not Ready",   "color": "#dc2626",
     "description": "Build consistent revenue for at least 6 months first."},
]

def get_score_band(score: int) -> dict:
    for band in SCORE_BANDS:
        if band["min"] <= score <= band["max"]:
            return band
    return SCORE_BANDS[-1]

# Quick sanity test
for test_score in [820, 700, 600, 500, 300]:
    band = get_score_band(test_score)
    print(f"  Score {test_score} → {band['label']}: {band['description']}")

### The 5 Scoring Signals

#### Each signal contributes 0-170 points. Total max=850

In [ ]:
# Signal 1: Revenue Consistency (0-170)
# Measures: Are revenues arriving on most days?
# Logic:    % of days with at least one credit transaction

def score_revenue_consistency(df: pd.DataFrame) -> dict:
    credits = df[df["type"] == "credit"]
    if credits.empty:
        return {"signal": "revenue_consistency", "score": 0, "max": 170,
                "detail": "No revenue transactions found.", "pct_active": 0}
        
    total_days  = max((df["date"].max() - df["date"].min()).days + 1, 1)
    active_days = credits["date"].nunique()
    pct_active  = active_days / total_days
    
    # 100% active -> 170, 67% -> 170*1.0 (capped), 50% -> ~113, <20% -> ~34
    raw_score = 170 * min(pct_active * 1.5, 1.0)
    score = int(round(raw_score))
    
    return {
        "signal": "revenue_consistency",
        "score": score, "max": 170,
        "active_days": active_days,
        "total_days": total_days,
        "pct_active": round(pct_active * 100, 1),
        "detail": f"Revenue on {active_days}/{total_days} days ({pct_active*100:.0f}% active)"
    }
    
    
# Signal 2: Cash Flow Health (0-170)
# Measures: Revenue-to-expense ratio

def score_cash_flow_health(df: pd.DataFrame) -> dict:
    total_revenue  = df[df["type"] == "credit"]["amount_ngn"].sum()
    total_expenses = df[df["type"] == "debit"]["amount_ngn"].sum()
    
    if total_revenue == 0:
        return {"signal": "cash_flow_health", "score": 0, "max": 170,
                "detail": "No revenue recorded.", "ratio": 0}
        
    ratio = total_revenue / (total_revenue + total_expenses) if (total_revenue + total_expenses) > 0 else 0
    score = int(round(170 * min(ratio * 1.2, 1.0)))
    
    return {
        "signal": "cash_flow_health",
        "score": score, "max": 170,
        "revenue": round(total_revenue, 2),
        "expenses": round(total_expenses, 2),
        "ratio": round(ratio, 3),
        "detail": f"Rev: ₦{total_revenue:,.0f} | Exp: ₦{total_expenses:,.0f} | Ratio: {ratio:.1%}"

    }
    
    
# Signal 3: Transaction Frequency (0-170)
# Measures: Average weekly transaction count

def score_transaction_frequency(df: pd.DataFrame) -> dict:
    credits = df[df["type"] == "credit"]
    if credits.empty:
        return {"signal": "transaction_frequency", "score": 0, "max": 170,
                "detail": "No transactions found.", "weekly_avg": 0}
        
    total_days = max((df["date"].max() - df["date"].min()).days + 1, 1)
    weeks      = total_days / 7
    weekly_avg = len(credits) / weeks
    
    if weekly_avg >= 7:
        score = 170
    elif weekly_avg >= 3:
        score = int(110 + (weekly_avg - 3) / 4 * 60)
    elif weekly_avg >= 1:
      score = int(70 + (weekly_avg - 1) / 2 * 40)
    else:
        score = int(70 * weekly_avg)
        
    return {
        "signal": "transaction_frequency",
        "score": min(score, 170), "max": 170,
        "total_transactions": len(credits),
        "weekly_avg": round(weekly_avg, 1),
        "detail": f"{len(credits)} transactions over {total_days} days ({weekly_avg:.1f}/week)"
    }
    
# Signal 4: Average Transaction Size (0-170)
# Measures: Typical deal size. Larger = more creditworthy.

def score_avg_transaction_size(df: pd.DataFrame) -> dict:
    credits = df[df["type"] == "credit"]
    if credits.empty:
        return {"signal": "avg_transaction_size", "score": 0, "max": 170,
                "detail": "No revenue transactions.", "avg_ngn": 0}
        
    avg = credits["amount_ngn"].mean()
    
    if avg >= 50_000:
        score = 170
    elif avg >= 20_000:
        score = int(130 + (avg - 20000) / 30000 * 40)
    elif avg >= 5_000:
        score = int(80 + (avg - 5000) / 15000 * 50)
    elif avg >= 1_000:
        score = int(20 + (avg - 1000) / 4000 * 60)
    else:
        score = int(20 * avg / 1000)
        
    return {
        "signal": "avg_transaction_size",
        "score": min(score, 170), "max": 170,
        "avg_ngn": round(avg, 2),
        "detail": f"Average transaction: ₦{avg:,.0f}"
    }
    
    
# Signal 5: Expenses Control (0-170)
# Measures: Operating expense ratio - lower spend-to-revenue = better

def score_expense_control(df: pd.DataFrame) -> dict:
    total_revenue  = df[df["type"] == "credit"]["amount_ngn"].sum()
    total_expenses = df[df["type"] == "debit"]["amount_ngn"].sum()
    
    if total_revenue == 0:
        return {"signal": "expense_control", "score": 0, "max": 170,
                "detail": "No revenue to measure against.", "expense_ratio": 1.0}
        
    expense_ratio = total_expenses / total_revenue
    
    if expense_ratio <= 0.3:
        score = 170
    elif expense_ratio <= 0.5:
        score = int(120 + (0.5 - expense_ratio) / 0.2 * 50)
    elif expense_ratio <= 0.8:
        score = int(60 + (0.8 - expense_ratio) / 0.3 * 60)
    elif expense_ratio <= 1.0:
        score = int(10 + (1.0 - expense_ratio) / 0.2 * 50)
    else:
        score = 10
    
    return {
        "signal": "expense_control",
        "score": min(score, 170), "max": 170,
        "expense_ratio": round(expense_ratio, 3),
        "total_expenses": round(total_expenses, 2),
        "detail": f"Expense ratio: {expense_ratio:.1%} (₦{total_expenses:,.0f} / ₦{total_revenue:,.0f})"
    }
    
print("All 5 scoring signal functions defined.")

### Unit Test All 5 Signals

In [ ]:
# Build a synthetic test DataFrame
from datetime import datetime, timedelta
import pandas as pd

base = datetime(2026, 4, 1)
test_rows = []

# 30 days of revenue (credits) - daily
for i in range(30):
    test_rows.append({"date": pd.Timestamp(base + timedelta(days=i)),
                      "amount_ngn": 8000 + i * 200, "type": "credit"})
    
# 4 expenses items (debits)
for i, amt in enumerate([45000, 12000, 25000, 8000]):
    test_rows.append({"date": pd.Timestamp(base + timedelta(days=i*7)),
                      "amount_ngn": amt, "type": "debit"})
    
test_df = pd.DataFrame(test_rows)

results = [
    score_revenue_consistency(test_df),
    score_cash_flow_health(test_df),
    score_transaction_frequency(test_df),
    score_avg_transaction_size(test_df),
    score_expense_control(test_df),
]


total = sum(r["score"] for r in results)
print(f"{'Signal':<30} {'Score':>6} / {'Max':>3}   Detail")
print("-" * 80)
for r in results:
    pct = r['score'] / r['max'] * 100
    bar = '█' * int(pct / 10) + '░' * (10 - int(pct / 10))
    print(f"{r['signal']:<30} {r['score']:>6} / {r['max']:>3}  {bar}  {r['detail'][:40]}")

print("-" * 80)
print(f"{'TOTAL':<30} {total:>6} / 850")
band = get_score_band(total)
print(f"\nBand: {band['label']} — {band['description']}")

### Composite Credit Score Calculator

In [ ]:
RECO_MAP = {
    "revenue_consistency": (
        "Accept payments every day, even on slow days. Consistent daily revenue "
        "is the #1 thing lenders look for."
    ),
    "cash_flow_health": (
        "Your expense ratio is high. Try to reduce non-essential spending this month "
        "to improve your revenue-to-cost ratio."
    ),
    "transaction_frequency": (
        "Increase your weekly transaction count. More customer payments, even small ones, "
        "signal an active healthy business."
    ),
    "avg_transaction_size": (
        "Try upselling or bundling products to increase your average transaction value. "
        "Higher ticket sizes improve creditworthiness."
    ),
    "expense_control": (
        "Audit your recurring expenses. Unnecessary debit transfers reduce your "
        "expense control score significantly."
    ),
}


def compute_credit_score(merchant_id: str, db_engine=None) -> dict:
    """
    Compute the full NombaIQ credit readiness score for a merchant.
    Returns score (0-850), band, all 5 signal breakdowns, and recommendations.
    """
    
    if db_engine is None:
        db_engine = engine
        
    df = pd.read_sql(
        text(
           "SELECT amount_ngn, type, category, channel, "
           "timestamp::date AS date FROM transactions "
           "WHERE merchant_id = :mid ORDER BY timestamp ASC" 
        ),
        db_engine, params={"mid": merchant_id} 
    )
    
    if df.empty:
        return {
            "merchant_id": merchant_id,
            "score": 0, "band": "Not Ready",
            "message": "No transaction history found",
            "signals": [], "recommendations": []
        }
        
    df["date"]       = pd.to_datetime(df["date"])
    df["amount_ngn"] = df["amount_ngn"].astype(float)
    
    # Compute all 5 signals
    signals = [
        score_revenue_consistency(df),
        score_cash_flow_health(df),
        score_transaction_frequency(df),
        score_avg_transaction_size(df),
        score_expense_control(df),
    ]
    
    total_score  = sum(s["score"] for s in signals)
    band_info    = get_score_band(total_score)
    
    # Top 3 weakest signals -> recommendations
    sorted_signals  = sorted(signals, key=lambda s: s["score"] / s["max"])
    recommendations = [
        {
            "signal":        s["signal"],
            "current_score": s["score"],
            "max_score":     s["max"],
            "advice":        RECO_MAP.get(s["signal"], "Improve this metric.")
        }
        for s in sorted_signals[:3]
    ] 
    
    return {
        "merchant_id":         merchant_id,
        "score":               total_score,
        "max_score":           850,
        "band":                band_info["label"],
        "band_color":          band_info["color"],
        "band_description":    band_info["description"],
        "computed_at":         datetime.now().isoformat(),
        "signals":             signals,
        "recommendations":     recommendations,
        "total_transactions":  len(df),
        "data_period_days":    max((df["date"].max() - df["date"].min()).days + 1, 1),
    }
    
    
# Run it for DEMO_001
result = compute_credit_score("DEMO_001")

print(f"{'='*52}")
print(f"  NOMBAIQ CREDIT READINESS SCORE")
print(f"  Merchant:   {result['merchant_id']}")
print(f"  Score:      {result['score']} / {result['max_score']}")
print(f"  Band:       {result['band']}")
print(f"  {result['band_description']}")
print(f"{'='*52}")
print()
print("Signal Breakdown:")
for s in result["signals"]:
    pct = s['score'] / s['max'] * 100
    bar = '█' * int(pct / 10) + '░' * (10 - int(pct / 10))
    print(f"  {s['signal']:<30} {bar}  {s['score']:>3}/{s['max']} — {s['detail'][:45]}")
print()
print("Top Recommendations:")
for i, rec in enumerate(result["recommendations"], 1):
    print(f"  {i}. [{rec['signal']}]")
    print(f"     {rec['advice']}")

### Visualize: Credit Score Gauge

In [ ]:
fig = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=result["score"],
    title={"text": f"Credit Readiness — <b>{result['band']}</b>", "font": {"size": 16}},
    number={"suffix": " / 850"},
    gauge={
        "axis": {"range": [0, 850], "tickwidth": 1},
        "bar": {"color": result["band_color"]},
        "steps": [
            {"range": [0,   449], "color": "#fecaca"},
            {"range": [449, 549], "color": "#fed7aa"},
            {"range": [549, 649], "color": "#fef08a"},
            {"range": [649, 749], "color": "#bbf7d0"},
            {"range": [749, 850], "color": "#86efac"},
        ],
        "threshold": {
            "line": {"color": "black", "width": 3},
            "thickness": 0.8,
            "value": result["score"]
        }
    }
))
fig.update_layout(height=350, template="plotly_white")
fig.show()

### Visualize: Signal Radar Chart

In [ ]:
signals     = result["signals"]
categories  = [s["signal"].replace("_", " ").title() for s in signals]
scores_pct  = [round(s["score"] / s["max"] * 100, 1) for s in signals]

# Close the loop for radar
categories_c = categories + [categories[0]]
scores_c     = scores_pct  + [scores_pct[0]]

fig = go.Figure(go.Scatterpolar(
    r=scores_c, theta=categories_c,
    fill="toself", name="Score %",
    line_color=result["band_color"],
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title="Credit Signal Radar — Score % per Dimension",
    height=400, template="plotly_white"
)
fig.show()

### Write `src/analytics/credit_score.py`

In [ ]:
os.makedirs("src/analytics", exist_ok=True)

credit_score_py = '''
import os
from datetime import datetime
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

SCORE_BANDS = [
    {"min": 750, "max": 850, "label": "Excellent",  "color": "#16a34a",
     "description": "Strong loan candidate. Ready for \u20a65M+ facility."},
    {"min": 650, "max": 749, "label": "Good",        "color": "#65a30d",
     "description": "Good candidate for \u20a61M-\u20a65M working capital loan."},
    {"min": 550, "max": 649, "label": "Fair",        "color": "#ca8a04",
     "description": "Eligible for micro-loan up to \u20a6500K."},
    {"min": 450, "max": 549, "label": "Developing",  "color": "#ea580c",
     "description": "3-6 months more transaction history needed."},
    {"min": 0,   "max": 449, "label": "Not Ready",   "color": "#dc2626",
     "description": "Build consistent revenue for at least 6 months first."},
]

RECO_MAP = {
    "revenue_consistency":   "Accept payments every day. Consistent daily revenue is the #1 thing lenders look for.",
    "cash_flow_health":      "Reduce non-essential spending to improve your revenue-to-cost ratio.",
    "transaction_frequency": "Increase weekly transaction count. More payments signal an active business.",
    "avg_transaction_size":  "Try upselling or bundling products to increase average transaction value.",
    "expense_control":       "Audit recurring expenses. Unnecessary debit transfers hurt this score.",
}


def get_score_band(score):
    for b in SCORE_BANDS:
        if b["min"] <= score <= b["max"]:
            return b
    return SCORE_BANDS[-1]


def _score_revenue_consistency(df):
    c = df[df["type"] == "credit"]
    if c.empty:
        return {"signal": "revenue_consistency", "score": 0, "max": 170, "detail": "No revenue.", "pct_active": 0}
    total  = max((df["date"].max() - df["date"].min()).days + 1, 1)
    active = c["date"].nunique()
    pct    = active / total
    return {"signal": "revenue_consistency", "score": int(round(170 * min(pct * 1.5, 1.0))),
            "max": 170, "active_days": active, "total_days": total,
            "pct_active": round(pct * 100, 1), "detail": f"Revenue on {active}/{total} days"}


def _score_cash_flow_health(df):
    rev = df[df["type"] == "credit"]["amount_ngn"].sum()
    exp = df[df["type"] == "debit"]["amount_ngn"].sum()
    if rev == 0:
        return {"signal": "cash_flow_health", "score": 0, "max": 170, "detail": "No revenue.", "ratio": 0}
    r = rev / (rev + exp) if (rev + exp) > 0 else 0
    return {"signal": "cash_flow_health", "score": int(round(170 * min(r * 1.2, 1.0))),
            "max": 170, "revenue": round(rev, 2), "expenses": round(exp, 2),
            "ratio": round(r, 3), "detail": f"Rev: \u20a6{rev:,.0f} Exp: \u20a6{exp:,.0f} Ratio: {r:.1%}"}


def _score_transaction_frequency(df):
    c = df[df["type"] == "credit"]
    if c.empty:
        return {"signal": "transaction_frequency", "score": 0, "max": 170, "detail": "No transactions.", "weekly_avg": 0}
    days = max((df["date"].max() - df["date"].min()).days + 1, 1)
    wkly = len(c) / (days / 7)
    s = (170 if wkly >= 7 else
         int(110 + (wkly - 3) / 4 * 60) if wkly >= 3 else
         int(70 + (wkly - 1) / 2 * 40) if wkly >= 1 else
         int(70 * wkly))
    return {"signal": "transaction_frequency", "score": min(s, 170), "max": 170,
            "total_transactions": len(c), "weekly_avg": round(wkly, 1),
            "detail": f"{len(c)} txns over {days} days ({wkly:.1f}/wk)"}


def _score_avg_transaction_size(df):
    c = df[df["type"] == "credit"]
    if c.empty:
        return {"signal": "avg_transaction_size", "score": 0, "max": 170, "detail": "No revenue.", "avg_ngn": 0}
    avg = c["amount_ngn"].mean()
    s = (170 if avg >= 50000 else
         int(130 + (avg - 20000) / 30000 * 40) if avg >= 20000 else
         int(80 + (avg - 5000) / 15000 * 50) if avg >= 5000 else
         int(20 + (avg - 1000) / 4000 * 60) if avg >= 1000 else
         int(20 * avg / 1000))
    return {"signal": "avg_transaction_size", "score": min(s, 170), "max": 170,
            "avg_ngn": round(avg, 2), "detail": f"Avg txn: \u20a6{avg:,.0f}"}


def _score_expense_control(df):
    rev = df[df["type"] == "credit"]["amount_ngn"].sum()
    exp = df[df["type"] == "debit"]["amount_ngn"].sum()
    if rev == 0:
        return {"signal": "expense_control", "score": 0, "max": 170, "detail": "No revenue.", "expense_ratio": 1.0}
    r = exp / rev
    s = (170 if r <= 0.3 else
         int(120 + (0.5 - r) / 0.2 * 50) if r <= 0.5 else
         int(60  + (0.8 - r) / 0.3 * 60) if r <= 0.8 else
         int(10  + (1.0 - r) / 0.2 * 50) if r <= 1.0 else 10)
    return {"signal": "expense_control", "score": min(s, 170), "max": 170,
            "expense_ratio": round(r, 3), "total_expenses": round(exp, 2),
            "detail": f"Expense ratio: {r:.1%}"}


def compute_credit_score(merchant_id: str, db_engine=None) -> dict:
    if db_engine is None:
        db_engine = create_engine(os.getenv("DATABASE_URL"))
    df = pd.read_sql(
        text("SELECT amount_ngn, type, category, channel, timestamp::date AS date "
             "FROM transactions WHERE merchant_id = :mid ORDER BY timestamp ASC"),
        db_engine, params={"mid": merchant_id}
    )
    if df.empty:
        return {"merchant_id": merchant_id, "score": 0, "band": "Not Ready",
                "message": "No transaction history.", "signals": [], "recommendations": []}
    df["date"]       = pd.to_datetime(df["date"])
    df["amount_ngn"] = df["amount_ngn"].astype(float)
    signals = [
        _score_revenue_consistency(df),
        _score_cash_flow_health(df),
        _score_transaction_frequency(df),
        _score_avg_transaction_size(df),
        _score_expense_control(df),
    ]
    total = sum(s["score"] for s in signals)
    band  = get_score_band(total)
    weak  = sorted(signals, key=lambda s: s["score"] / s["max"])[:3]
    recos = [{"signal": s["signal"], "current_score": s["score"], "max_score": s["max"],
              "advice": RECO_MAP.get(s["signal"], "")} for s in weak]
    return {
        "merchant_id": merchant_id, "score": total, "max_score": 850,
        "band": band["label"], "band_color": band["color"],
        "band_description": band["description"],
        "computed_at": datetime.now().isoformat(),
        "signals": signals, "recommendations": recos,
        "total_transactions": len(df),
        "data_period_days": max((df["date"].max() - df["date"].min()).days + 1, 1),
    }
'''

with open("src/analytics/credit_score.py", "w") as f:
    f.write(credit_score_py.strip())

print( "src/analytics/credit_score.py written.")

## Add Credit Score Endpoint to FastAPI

In [ ]:
credit_route = '''
# ── Add to src/api/main.py ─────────────────────────────────────────────

@app.get("/merchant/{merchant_id}/credit-score")
async def get_credit_score(merchant_id: str):
    from src.analytics.credit_score import compute_credit_score
    result = compute_credit_score(merchant_id, ENGINE)
    if not result.get("signals"):
        raise HTTPException(status_code=404, detail=f"No data for {merchant_id}")
    return result
'''

with open("src/api/credit_route.py", "w") as f:
    f.write(credit_route.strip())

print(" src/api/credit_route.py written.")
print("   Paste this endpoint into src/api/main.py")

## Done Test

In [ ]:
# ── THE DAY 4 DONE TEST ──────────────────────────────────────────────────
result = compute_credit_score("DEMO_001")

checks = {
    "score is integer 0–850":         isinstance(result["score"], int) and 0 <= result["score"] <= 850,
    "band is valid string":           result["band"] in [b["label"] for b in SCORE_BANDS],
    "exactly 5 signals returned":     len(result["signals"]) == 5,
    "all signals have score + max":   all("score" in s and "max" in s for s in result["signals"]),
    "all signal scores non-negative": all(s["score"] >= 0 for s in result["signals"]),
    "total equals sum of signals":    result["score"] == sum(s["score"] for s in result["signals"]),
    "recommendations present":        len(result.get("recommendations", [])) > 0,
    "band_description present":       len(result.get("band_description", "")) > 0,
    "credit_score.py exists":         os.path.exists("src/analytics/credit_score.py"),
}

all_passed = all(checks.values())
for check, passed in checks.items():
    print(f"  {'Passed' if passed else 'Failed'} {check}")

print()
if all_passed:
    print(f" DAY 4 DONE TEST PASSED")
    print(f"   Score: {result['score']}/850 — {result['band']}")
    print(f"   {result['band_description']}")
else:
    print(" Some checks failed — review the cells above.")

print()
print(" Day 4 complete. Move to Day 5: AI Advisor.")